# Phase 3 — Ablation Study and Hate Speech Benchmark

Two goals:
1. Quantify how much each feature group contributes (A1 text-only → A4 full model)
2. Test text-only baselines on the Automated Hate Speech dataset (3-class)


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError("Cannot find F21MP project root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.phase3_ablation_benchmark.evaluation.ablation import run_ablation, run_hate_speech_benchmark

## 1. Ablation A1–A4

Same 80/20 split as Phase 1, same data — just different feature sets.


In [2]:
ablation_out = run_ablation(save_outputs=True)
ablation_summary = ablation_out['summary']
ablation_summary

,experiment,accuracy,precision,recall,f1,roc_auc
0,A1_text_only,0.965992,0.000000,0.0,0.000000,0.509497
1,A2_text_plus_peerness,0.991093,0.540541,0.8,0.645161,0.995419
2,A3_text_plus_repetition,0.943320,0.040000,0.2,0.066667,0.686020
3,A4_full,0.999595,0.961538,1.0,0.980392,0.999984
4,A5_full_plus_demographics,0.999595,0.961538,1.0,0.980392,0.999984


## 2. Ablation F1 bar chart


In [3]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(ablation_summary['experiment'], ablation_summary['f1'], color=['#1f77b4','#2ca02c','#ff7f0e','#9467bd'])
ax.set_title('Phase 3 Ablation (A1–A4): F1')
ax.set_ylabel('F1')
ax.set_xlabel('Experiment')
for i, v in enumerate(ablation_summary['f1']):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylim(0, min(1.05, max(ablation_summary['f1']) + 0.12))
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

/var/folders/qd/h9jkgddx56z_tcs9xy5b24fw0000gn/T/ipykernel_73240/1331737698.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Hate Speech benchmark (3-class)

Davidson et al. labeled data: hate speech vs offensive vs neither.
Comparing TF-IDF + Logistic Regression vs TF-IDF + XGBoost.


In [4]:
hs_out = run_hate_speech_benchmark(save_outputs=True)
hs_summary = hs_out['summary']
hs_summary

/Users/sagarmamindla/Desktop/F21MP/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,n_samples
0,tfidf_logreg,0.892677,0.807038,0.627424,0.656370,0.874876,4957
1,tfidf_xgboost,0.910026,0.775098,0.716196,0.724195,0.901194,4957


## 4. Results

**Ablation:**
- A1 (text only): F1 = 0.000 — completely fails on its own with this imbalance
- A2 (+ peerness): F1 = 0.645 — large jump, peerness is clearly important
- A3 (+ repetition): F1 = 0.042 — helps a little but not much alone
- A4 (full): F1 = 0.980 — combining both gives the strongest result

**Hate Speech:**
XGBoost outperforms logistic regression on macro F1 (0.724 vs 0.656), 
but both are well below in-domain A4 — consistent with cross-domain difficulty.
